# kazeval on Kaggle — canonical GPU runs (compute policy 2026-07-04)

One-time Kaggle setup:
1. **Settings → Accelerator**: GPU T4 x2 or P100 (both 16 GB, fp16 — no bf16).
2. **Settings → Internet**: ON (needs a phone-verified account).
3. **Add-ons → Secrets**: add `HF_TOKEN` = a HuggingFace token of an account that has
   accepted the conditions of the gated datasets/models used below
   ([issai/kazqad-retrieval](https://huggingface.co/datasets/issai/kazqad-retrieval),
   and [inceptionai/Llama-3.1-Sherkala-8B-Chat](https://huggingface.co/inceptionai/Llama-3.1-Sherkala-8B-Chat) for the KazMMLU ceiling run).

The repo is cloned from GitHub `main` — **push before launching**, Kaggle sees only what is pushed.
Free quota: 30 GPU-h/week, sessions ≤ 12 h — long runs must checkpoint/finish inside one session.


In [ ]:
!git clone --depth 1 https://github.com/kekeront/qymyzlm.git /kaggle/working/qymyzlm
%pip install -q /kaggle/working/qymyzlm/evallab

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

login(UserSecretsClient().get_secret("HF_TOKEN"))

## Run 1 — KazQAD retrieval + reranking planka (mE5-large)

Full-corpus retrieval (1,929 test queries vs the 825,309-passage corpus) + the deterministic
`kazqad-hardneg-bm25-v1` reranking. Expected ~1-3 h on P100/T4 at batch 64.
Writes one committed-style JSON record per (task, split) into `/kaggle/working/results/`.


In [ ]:
!python -m kazeval.run_retrieval \
    --model intfloat/multilingual-e5-large \
    --batch-size 64 \
    --output-dir /kaggle/working/results

## Run 2 (optional) — KazMMLU 3-shot, Sherkala-Chat-8B in 4-bit (W5: same-protocol ceiling)

8B in 4-bit ≈ 5.5 GB weights — fits a single T4/P100. Uncomment to run.


In [ ]:
# %pip install -q bitsandbytes
# !python -m kazeval.run_kazmmlu \
#     --model inceptionai/Llama-3.1-Sherkala-8B-Chat \
#     --model-args dtype=float16,load_in_4bit=True \
#     --batch-size auto \
#     --output-dir /kaggle/working/results


## Collect results

Download every JSON under `/kaggle/working/results/` (Output tab), drop the files into
`evallab/results/` in the repo, then locally:

```bash
python -m kazeval.leaderboard   # re-renders the README leaderboard
```


In [ ]:
import pathlib

for p in sorted(pathlib.Path("/kaggle/working/results").glob("*.json")):
    print(p)
    print(p.read_text())